In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import torch.nn as nn
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score, f1_score, confusion_matrix
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
from tqdm import tqdm

import sys
sys.path.append('../../src')

from preprocessing import *
from models import *
from utils import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
plt.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 20,
    'axes.labelsize': 15,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 14,
    'figure.titlesize': 18,
    'figure.dpi': 300,
    'savefig.dpi': 300,
})
plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['xtick.major.width'] = 1.5
plt.rcParams['ytick.major.width'] = 1.5
plt.rcParams['lines.markersize'] = 8

In [ ]:
dfs = get_dfs(os.path.dirname(os.path.dirname(os.getcwd())))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)

notes = create_notes_df(dfs, filename='../../data/embeddings/emb_med_gte_simcse_en_ger.npy')

biopsy_df = dfs['biopsy']

all_valid_patient_ids = get_valid_patient_ids(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    min_ts_count=10,
    require_notes=True,
)

with open('../../data/splits/pool_assignments.json') as f:
    pool_assignments = json.load(f)
pool_a_ids = np.asarray(pool_assignments['pool_a'])
pool_a_set = set(pool_a_ids.tolist())
selected_patient_ids = np.asarray([pid for pid in all_valid_patient_ids if pid in pool_a_set])

backbone_split_path = '../../data/splits/global_split_pool_a_9010.json'
backbone_split_ids = get_or_create_global_split(
    patient_ids=selected_patient_ids,
    split_json_path=backbone_split_path,
    train_size=0.9,
    val_size=0.1,
    test_size=0.0,
    random_state=42,
    shuffle=True,
    force_recreate=False,
)

preprocessing_ref = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=backbone_split_ids['train'],
    fit_preprocessing=True,
    min_ts_count=10,
    require_notes=True,
)
preprocessing_artifacts = preprocessing_ref.preprocessing_artifacts

all_dataset = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=selected_patient_ids,
    preprocessing_artifacts=preprocessing_artifacts,
    fit_preprocessing=False,
    min_ts_count=10,
    require_notes=True,
)

print(f"Pool A size: {len(selected_patient_ids)}")
print(f"All-patient dataset size: {len(all_dataset)}")
print(f"Preprocessing fit on backbone train split ({len(backbone_split_ids['train'])} patients)")

In [ ]:
batch_size = 16
all_dataloader = DataLoader(all_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(
    att_encoder,
    categorical_cardinalities=preprocessing_ref.categorical_cardinalities,
    use_static=True,
    use_notes=True,
).to(device)

model_path = '../../models/backbone_poola_9010_best.pt'
checkpoint = torch.load(model_path, weights_only=False, map_location=device)
model.load_state_dict(checkpoint)
print(f"Loaded backbone from {model_path}")

In [ ]:
def extract_horizon_reprs(
    dataloader, model, horizons, label_key, rel_days_key=None,
    min_history_days=90, max_days=180, max_samples_per_patient=100,
    sampling_strategy="uniform", is_training_set=False, random_state=42,
):
    device = next(model.parameters()).device
    model.eval()

    hr_repr = {H: [] for H in horizons}
    hr_label = {H: [] for H in horizons}
    hr_days = {H: [] for H in horizons}
    hr_pids = {H: [] for H in horizons}

    all_pids = set()
    positive_pids = set()

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting Timesteps"):
            pid = batch['patient_id']
            slen = batch['seq_len']

            cat_static = batch['static_categorical_features'].to(device)
            num_static = batch['static_numerical_features'].to(device)
            full_ts = batch['ts_features'].to(device)
            timesteps = batch['timesteps'].to(device)
            mask_ = batch['mask'].to(device)
            value_mask_full = batch['value_mask'].to(device)

            raw_labels_data = batch[label_key]
            labels_data = []
            for val in raw_labels_data:
                if isinstance(val, torch.Tensor):
                    val = float(val.item()) if val.numel() == 1 else val.cpu().numpy()
                labels_data.append(val)

            if rel_days_key and rel_days_key in batch:
                raw_rel_days_data = batch[rel_days_key]
                rel_days_data = []
                for dval in raw_rel_days_data:
                    if isinstance(dval, torch.Tensor):
                        dval = float(dval.item()) if dval.numel() == 1 else dval.cpu().numpy()
                    rel_days_data.append(dval)
                rel_days_data = np.array(rel_days_data)
            else:
                rel_days_data = None

            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps = batch['notes_timesteps'].to(device)
            notes_mask = batch['notes_mask'].to(device)

            B, T, F = full_ts.shape

            for i in range(B):
                patient_id_i = pid[i]
                all_pids.add(patient_id_i)
                label_or_list = labels_data[i]

                if isinstance(label_or_list, (int, float, np.number)):
                    if label_or_list == 1:
                        positive_pids.add(patient_id_i)
                elif isinstance(label_or_list, (list, np.ndarray)):
                    if len(label_or_list) > 0:
                        positive_pids.add(patient_id_i)

                if slen[i] < 2:
                    continue

                seq_len_i = slen[i].item()
                ts_i = full_ts[i:i+1, :seq_len_i, :]
                tm_i = timesteps[i:i+1, :seq_len_i]
                mk_i = mask_[i:i+1, :seq_len_i]

                inp_seq = ts_i[:, :-1, :]
                inp_mask = mk_i[:, :-1]
                elapsed_times = build_elapsed_times(tm_i[:, :-1], inp_mask)
                inp_value_mask = value_mask_full[i:i+1, :seq_len_i-1, :]

                notes_emb_i = notes_embeddings[i:i+1]
                notes_ts_i = notes_timesteps[i:i+1]
                notes_mk_i = notes_mask[i:i+1]

                out, lstm_out, _, _ = model(
                    x=inp_seq,
                    elapsed_times=elapsed_times,
                    timesteps=tm_i[:, :-1],
                    notes_embeddings=notes_emb_i,
                    notes_timesteps=notes_ts_i,
                    static_features=(cat_static[i:i+1], num_static[i:i+1]),
                    mask=inp_mask,
                    notes_mask=notes_mk_i,
                    value_mask=inp_value_mask,
                )

                time_arr = tm_i.cpu().numpy().flatten()

                for k in range(seq_len_i - 1):
                    cur_day = time_arr[k]
                    if cur_day < min_history_days or cur_day > max_days:
                        continue

                    rep_ = lstm_out[0, k, :].cpu().numpy()

                    for H in horizons:
                        if rel_days_data is not None:
                            event_label = label_or_list
                            event_day = rel_days_data[i]
                            label_ = 1 if (event_label == 1) and (0 < (event_day - cur_day) <= H) else 0
                        else:
                            if isinstance(label_or_list, (list, np.ndarray)) and len(label_or_list) > 0:
                                label_ = int(any(0 < (d - cur_day) <= H for d in label_or_list))
                            else:
                                label_ = 0

                        hr_repr[H].append(rep_)
                        hr_label[H].append(label_)
                        hr_days[H].append(cur_day)
                        hr_pids[H].append(patient_id_i)

    # Priority sampling (same as classification.ipynb) 
    # I removed this for kfold but maybe for 70-10-20 training it's required.
    if max_samples_per_patient is not None and max_samples_per_patient > 0:
        rng = np.random.default_rng(random_state)
        for H in horizons:
            if len(hr_pids[H]) == 0:
                continue
            pids_arr = np.array(hr_pids[H])
            labels_arr = np.array(hr_label[H])
            keep_mask = np.zeros(len(pids_arr), dtype=bool)

            for pid_i in np.unique(pids_arr):
                idx = np.where(pids_arr == pid_i)[0]
                if len(idx) <= max_samples_per_patient:
                    chosen = list(idx)
                elif is_training_set:
                    pos_idx = idx[labels_arr[idx] == 1]
                    neg_idx = idx[labels_arr[idx] == 0]
                    chosen = []
                    if len(pos_idx) > 0:
                        if len(pos_idx) > max_samples_per_patient:
                            chosen.extend(rng.choice(pos_idx, size=max_samples_per_patient, replace=False))
                        else:
                            chosen.extend(pos_idx)
                    remaining = max_samples_per_patient - len(chosen)
                    if remaining > 0 and len(neg_idx) > 0:
                        safe = min(remaining, len(neg_idx))
                        if sampling_strategy == "uniform":
                            pos = np.linspace(0, len(neg_idx) - 1, num=safe, dtype=int)
                            chosen.extend(neg_idx[pos])
                        elif sampling_strategy == "random":
                            chosen.extend(rng.choice(neg_idx, size=safe, replace=False))
                        else:
                            chosen.extend(neg_idx[:safe])
                else:
                    if sampling_strategy == "uniform":
                        pos = np.linspace(0, len(idx) - 1, num=max_samples_per_patient, dtype=int)
                        chosen = list(idx[pos])
                    elif sampling_strategy == "random":
                        chosen = list(rng.choice(idx, size=max_samples_per_patient, replace=False))
                    else:
                        chosen = list(idx[:max_samples_per_patient])

                keep_mask[np.array(chosen, dtype=int)] = True

            kept_idx = np.where(keep_mask)[0]
            hr_repr[H] = [hr_repr[H][j] for j in kept_idx]
            hr_label[H] = [hr_label[H][j] for j in kept_idx]
            hr_days[H] = [hr_days[H][j] for j in kept_idx]
            hr_pids[H] = [hr_pids[H][j] for j in kept_idx]

    for H in horizons:
        hr_repr[H] = np.array(hr_repr[H])
        hr_label[H] = np.array(hr_label[H])
        hr_days[H] = np.array(hr_days[H])
        hr_pids[H] = np.array(hr_pids[H])

    for H in horizons:
        unique_pids, counts = np.unique(hr_pids[H], return_counts=True) if len(hr_pids[H]) else (np.array([]), np.array([]))
        print(f"Horizon {H}: {len(unique_pids)} patients, avg {np.mean(counts):.1f} samples/patient, max {int(np.max(counts))} samples/patient")

    print(f"Total unique patients: {len(all_pids)}, with event: {len(positive_pids)}")
    return hr_repr, hr_label, hr_days, hr_pids

In [ ]:
horizons = [30, 90, 180]
min_history_days = 90
max_days = 720
max_samples_per_patient = 100
sampling_strategy = "uniform"

print("Extracting representations for all Pool A patients...\n")

all_graft_repr, all_graft_lbl, all_graft_days, all_graft_pids = extract_horizon_reprs(
    all_dataloader, model, horizons, "graft_loss_label", "loss_rel_days",
    min_history_days, max_days,
    max_samples_per_patient=max_samples_per_patient,
    sampling_strategy=sampling_strategy, is_training_set=False,
)

all_rej_repr, all_rej_lbl, all_rej_days, all_rej_pids = extract_horizon_reprs(
    all_dataloader, model, horizons, "rej_rel_days", None,
    min_history_days, max_days,
    max_samples_per_patient=max_samples_per_patient,
    sampling_strategy=sampling_strategy, is_training_set=False,
)

all_mort_repr, all_mort_lbl, all_mort_days, all_mort_pids = extract_horizon_reprs(
    all_dataloader, model, horizons, "death_label", "death_rel_days",
    min_history_days, max_days,
    max_samples_per_patient=max_samples_per_patient,
    sampling_strategy=sampling_strategy, is_training_set=False,
)

for H in horizons:
    print(f"\n===== Extracted Features for Horizon {H} days =====")
    print(f"Graft Loss: {all_graft_repr[H].shape}, pos rate: {all_graft_lbl[H].mean():.4f}")
    print(f"Rejection:  {all_rej_repr[H].shape}, pos rate: {all_rej_lbl[H].mean():.4f}")
    print(f"Mortality:  {all_mort_repr[H].shape}, pos rate: {all_mort_lbl[H].mean():.4f}")

In [ ]:
clf_dir = '../../models/run-16-final'

tasks = {
    'GraftLoss': (all_graft_repr, all_graft_lbl, all_graft_days, all_graft_pids),
    'Rejection': (all_rej_repr, all_rej_lbl, all_rej_days, all_rej_pids),
    'Mortality': (all_mort_repr, all_mort_lbl, all_mort_days, all_mort_pids),
}

n_splits = 5
skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

# For each task/horizon: replay the same CV folds, load the saved MLP,
# and collect val-set predicted probabilities (out-of-fold predictions).
oof_results = {}

for task_name, (repr_dict, lbl_dict, days_dict, pids_dict) in tasks.items():
    for H in horizons:
        key = f"{task_name}@{H}"
        save_path = os.path.join(clf_dir, f"{key}_clf.pth")

        ckpt = torch.load(save_path, weights_only=False, map_location=device)
        best_fold_idx = ckpt['best_fold']

        X_all = repr_dict[H]
        y_all = lbl_dict[H]
        pids_all = pids_dict[H]

        pid_to_label = {}
        for pid in np.unique(pids_all):
            pid_to_label[pid] = int(y_all[pids_all == pid].any())
        strat_y = np.array([pid_to_label[p] for p in pids_all])

        # Replay the exact same fold split to get the val indices for the best fold
        for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, strat_y, groups=pids_all)):
            if fold == best_fold_idx:
                break

        X_val = X_all[val_idx]
        y_val = y_all[val_idx]
        pids_val = pids_all[val_idx]

        mlp = SimpleMLP(input_dim=X_val.shape[1]).to(device)
        mlp.load_state_dict(ckpt['model_state_dict'])
        mlp.eval()

        with torch.no_grad():
            X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
            logits = mlp(X_val_t)
            probs = torch.sigmoid(logits).cpu().numpy()

        oof_results[key] = {
            'y_true': y_val,
            'y_prob': probs,
            'pids': pids_val,
            'fold_metrics': ckpt.get('fold_metrics', []),
            'cv_metrics': ckpt.get('cv_metrics', {}),
        }

        auc = roc_auc_score(y_val.astype(int), probs) if len(set(y_val.astype(int))) > 1 else float('nan')
        print(f"{key} (fold {best_fold_idx}): {len(y_val)} val samples, "
              f"{int(y_val.sum())} pos, AUC={auc:.4f}")

In [ ]:
def calibration_analysis(y_true, y_prob, n_bins=10):
    """Compute calibration metrics: original, Platt-scaled, and isotonic."""
    results = {}

    # Original
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins)
    results['original'] = {
        'prob_true': prob_true,
        'prob_pred': prob_pred,
        'brier': brier_score_loss(y_true, y_prob),
        'probs': y_prob,
    }

    # Platt scaling
    platt = LogisticRegression(C=1.0, solver='lbfgs')
    platt.fit(y_prob.reshape(-1, 1), y_true)
    platt_probs = platt.predict_proba(y_prob.reshape(-1, 1))[:, 1]
    prob_true_p, prob_pred_p = calibration_curve(y_true, platt_probs, n_bins=n_bins)
    results['platt'] = {
        'prob_true': prob_true_p,
        'prob_pred': prob_pred_p,
        'brier': brier_score_loss(y_true, platt_probs),
        'probs': platt_probs,
        'scaler': platt,
    }

    # Isotonic regression
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(y_prob, y_true)
    iso_probs = iso.predict(y_prob)
    prob_true_i, prob_pred_i = calibration_curve(y_true, iso_probs, n_bins=n_bins)
    results['isotonic'] = {
        'prob_true': prob_true_i,
        'prob_pred': prob_pred_i,
        'brier': brier_score_loss(y_true, iso_probs),
        'probs': iso_probs,
        'scaler': iso,
    }

    return results

In [ ]:
def plot_calibration(cal_results, title, save_path=None, n_bins=10):
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # --- Left: Calibration curves ---
    ax = axes[0]
    ax.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')

    colors = {'original': 'blue', 'platt': 'red', 'isotonic': 'green'}
    labels = {'original': 'Original', 'platt': 'Platt Scaled', 'isotonic': 'Isotonic'}

    for method in ['original', 'platt', 'isotonic']:
        r = cal_results[method]
        ax.plot(
            r['prob_pred'], r['prob_true'], '-o',
            color=colors[method],
            label=f"{labels[method]} (Brier: {r['brier']:.4f})",
        )

    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('True Probability (Fraction of Positives)')
    ax.set_title(f'{title} — Calibration Curves')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

    # --- Right: Probability distribution histogram ---
    ax2 = axes[1]
    bins = np.linspace(0, 1, n_bins + 1)

    for method in ['original', 'platt', 'isotonic']:
        ax2.hist(
            cal_results[method]['probs'], bins=bins,
            alpha=0.35, color=colors[method], label=labels[method],
            edgecolor='black', linewidth=0.5,
        )

    ax2.set_xlabel('Predicted Probability')
    ax2.set_ylabel('Count')
    ax2.set_title(f'{title} — Probability Distribution')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()
    return fig

In [ ]:
n_bins = 15
all_cal_results = {}

for key, data in oof_results.items():
    y_true = data['y_true'].astype(int)
    y_prob = data['y_prob']

    cal = calibration_analysis(y_true, y_prob, n_bins=n_bins)
    all_cal_results[key] = cal

    save_path = f"../../plots/calibration_{key.replace('@', '_')}.png"
    plot_calibration(cal, title=key, save_path=save_path, n_bins=n_bins)

In [ ]:
print(f"{'Event':<20} {'Brier (Orig)':>14} {'Brier (Platt)':>15} {'Brier (Iso)':>14} {'AUC':>8}")
print("=" * 75)

for key in sorted(all_cal_results.keys()):
    cal = all_cal_results[key]
    data = oof_results[key]
    auc = roc_auc_score(data['y_true'].astype(int), data['y_prob'])
    print(
        f"{key:<20} "
        f"{cal['original']['brier']:>14.4f} "
        f"{cal['platt']['brier']:>15.4f} "
        f"{cal['isotonic']['brier']:>14.4f} "
        f"{auc:>8.4f}"
    )

In [ ]:
def plot_threshold_analysis(y_true, y_prob, title, save_path=None):
    thresholds = np.linspace(0.01, 0.99, 200)
    sensitivities = []
    specificities = []
    f1s = []
    precisions = []

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        if y_pred.sum() == 0 or y_pred.sum() == len(y_pred):
            sensitivities.append(np.nan)
            specificities.append(np.nan)
            f1s.append(np.nan)
            precisions.append(np.nan)
            continue
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        sensitivities.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
        f1s.append(f1_score(y_true, y_pred, zero_division=0))
        precisions.append(tp / (tp + fp) if (tp + fp) > 0 else 0)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(thresholds, sensitivities, label='Sensitivity (Recall)', color='blue')
    ax.plot(thresholds, specificities, label='Specificity', color='red')
    ax.plot(thresholds, f1s, label='F1 Score', color='green')
    ax.plot(thresholds, precisions, label='Precision', color='orange')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title(f'{title} — Threshold Analysis')
    ax.legend(loc='center left')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])

    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()
    return fig

for key, data in oof_results.items():
    save_path = f"../../plots/threshold_{key.replace('@', '_')}.png"
    plot_threshold_analysis(
        data['y_true'].astype(int),
        data['y_prob'],
        title=key,
        save_path=save_path,
    )

In [ ]:
# After calibration, pick clinically useful operating points.
# For medical data: high sensitivity is typically preferred (don't miss events).
# The calibrated probabilities let you set a threshold where predictions below it
# can be confidently dismissed as low-risk.

print(f"{'Event':<20} {'Threshold':>10} {'Sens':>8} {'Spec':>8} {'Prec':>8} {'F1':>8} {'Method':>10}")
print("=" * 80)

for key in sorted(oof_results.keys()):
    data = oof_results[key]
    cal = all_cal_results[key]
    y_true = data['y_true'].astype(int)

    # Use the calibration method with the lowest Brier score
    best_method = min(['original', 'platt', 'isotonic'], key=lambda m: cal[m]['brier'])
    probs = cal[best_method]['probs']

    # Find threshold that gives >= 80% sensitivity
    best_thr = 0.5
    best_f1 = -1
    for thr in np.linspace(0.01, 0.99, 500):
        y_pred = (probs >= thr).astype(int)
        if y_pred.sum() == 0:
            continue
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        if sens >= 0.80:
            f1 = f1_score(y_true, y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thr = thr

    y_pred = (probs >= best_thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"{key:<20} {best_thr:>10.3f} {sens:>8.3f} {spec:>8.3f} {prec:>8.3f} {f1:>8.3f} {best_method:>10}")